# Assignment 5: Build Your Own Persona Agent

In Lab 4 you built **Study Buddy** — a quiz agent that studies a short set of notes, asks you
questions about them one at a time, and logs anything you get wrong to a file so you can review
it later.

In this assignment you'll build the **exact same kind of agent**, but with your own persona and
on a topic of your own choosing instead of Agentic AI fundamentals. It can be material from
another class, a hobby, a certification you're studying for, a language you're learning, anything
you can write a short set of notes about and ask factual questions on.

You'll reuse your own Lab 4 code at almost every step. This assignment is about **adapting** what
you already built, not writing it from scratch.

**What you'll build:**
1. A persona — a name and personality for your agent
2. A short set of study notes on a topic of your choice, read from a PDF file
3. A system prompt that combines your persona and your notes
4. A `chat(message, history)` function
5. A Gradio chat interface
6. One custom tool your Persona can call
7. A full agent loop that can call your tool as many times as needed

**Provider:** Azure OpenAI via APIM, deployment `gpt-5.4-ptu` — same as Lab 4.

There's nothing difficult here, if you can find where you did each step in Lab 4, you can do it
here too.

## 0. Install the packages (run once)

Same packages as Lab 4 .

In [1]:
!uv pip install --upgrade openai pypdf gradio python-dotenv

Resolved 59 packages in 5.95s
 Downloaded openai
Prepared 8 packages in 39.52s
Uninstalled 8 packages in 14.41s
Installed 8 packages in 18.69s
 - filelock==4.0.1
 + filelock==4.0.6
 - httpcore2==2.13.0
 + httpcore2==2.13.1
 - httpx2==2.13.0
 + httpx2==2.13.1
 - huggingface-hub==1.32.0
 + huggingface-hub==1.33.0
 - openai==3.18.0
 + openai==3.20.0
 - pytz==2026.3.post1
 + pytz==2026.4
 - starlette==1.6.0
 + starlette==1.7.0
 - uvicorn==0.53.0
 + uvicorn==0.54.0


## 1. Imports

**TODO:** Copy the imports you used in Lab 4: `load_dotenv`, `os`, `sys`, `json`, `PdfReader`
(from `pypdf`), `gr` (gradio), and `AzureOpenAI`.

**Hint:** look at Lab 4, Step 1.

In [2]:
from dotenv import load_dotenv
import os
import sys
import json

from openai import AzureOpenAI
from pypdf import PdfReader
from IPython.display import Markdown, display
import gradio as gr

c:\Users\colby\Desktop\IT3025C Labs\Lab 4\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 2. Connect to Azure OpenAI via APIM

This part is identical to Lab 4 — same `.env` file, same deployment. You can copy this cell
directly from Lab 4, Step 2.

Your `.env` file should contain:

```
AZURE_OPENAI_ENDPOINT=https://your-resource-name.openai.azure.com/
AZURE_OPENAI_API_KEY=your-apim-key-here
```

In [3]:
load_dotenv(override=True)

azure_endpoint = os.getenv("AZURE_OPENAI_ENDPOINT")
azure_api_key = os.getenv("AZURE_OPENAI_API_KEY")

if not all([azure_endpoint, azure_api_key]):
    sys.exit("Missing settings. Set AZURE_OPENAI_ENDPOINT and AZURE_OPENAI_API_KEY in your .env file.")

print(f"Azure OpenAI key exists and begins {azure_api_key[:8]}")

# The deployment we're using for this whole assignment — fixed, do not change it
DEPLOYMENT = "gpt-5.4-ptu"

azure_client = AzureOpenAI(
    azure_endpoint=azure_endpoint,
    api_key=azure_api_key,
    api_version="2024-10-21",
)

Azure OpenAI key exists and begins 512e1c13


## 3. Choose your persona and topic, then load your notes from a PDF

Instead of the Agentic AI notes from Lab 4, you'll give your agent its own persona and its own
topic.

**TODO:** Fill in two things below:

- `persona` — a short description of your agent's name and personality, the same way Lab 4
  described Study Buddy. A sentence or two is plenty: who is it, and how does it talk?
- Your study notes as a **PDF file**. Write a few short paragraphs or a bulleted list of facts on
  a topic of your choice, save it as a PDF, and place it at `notes/notes.pdf` next to this
  notebook — exactly the same layout Lab 4 used. The cell below reads the text out of that PDF
  into a `notes` variable for you, the same way Lab 4, Step 4 did.

**Example persona:** "Coach Rowan, a blunt, no-nonsense trainer who keeps feedback short and
direct, but always tells you exactly what to fix."

**Example notes topic (a language-learning topic):**
> "The present tense of a regular -ar verb in Spanish drops the -ar and adds -o, -as, -a, -amos,
> -áis, -an. For example, hablar (to speak) becomes hablo, hablas, habla, hablamos, habláis,
> hablan. ..."

In [ ]:
persona = (
    "Finance Coach, a calm and practical study assistant who helps studnet review "
    "liquidity, working capital, cash management, and breakeven analysis Finance Coach "
    " asks focused questions, explains formulas clearly, and helps the student identify topics they should review again."
)

reader = PdfReader("Notes/module5_notes.pdf")
notes = "" 
for page in reader.pages:
    text = page.extract_text()
    if text:
        notes += text

print(persona)
print(notes[:500], "...")

Finance Coach, a calm and practical study assistant who helps studnet review liquidity, working capital, cash management, and breakeven analysis Finance Coach  asks focused questions, explains formulas clearly, and helps the student identify topics they should review again.
Module 5 Study Notes: Liquidity, Working Capital, and Breakeven
This study sheet summarizes Module 06: Liquidity, Working Capital, and Breakeven. The
main ideas are how entrepreneurs measure short-term financial health, manage cash and
working capital, and calculate the revenue level needed for survival or operating
breakeven.
Financial Ratios and Liquidity
Financial ratio analysis uses relationships among financial statement accounts to judge
a company’s condition and performance. Ratios can he ...


## 4. Build your system prompt

**TODO:** Build a `system_prompt` f-string, the same way you did in Lab 4, Step 5 — combining
your `persona` and your `notes` variables, and your own tool's name in the "record a missed
question" rule (you'll design that tool in Step 8, below).

**Hint:** Look at Lab 4's system prompt for the shape to copy: a "Your role" section built from
`persona`, a "Context" section with your `notes`, and a "Rules" section covering one question at
a time, feedback on the answer, and recording missed questions.

In [5]:
system_prompt = f"""
# Your role

Your are {persona}

#Context 

Here are the study notes you should use:

{notes}

# Rules
Asl exactly one question at a time based only on the notes.

Wait for the student's answer before askign another question.

After the student answers, clearly say whether the answer is correct or incorrect, then give a vreif explanation.

If the student gets a question wrong, says "I don't know", or seems unsure, use your tool to record the topic or question they should review.

Do not ask questions about material that is not in the notes.

Keep your tone calm, practical, and encouraging

"""

display(Markdown(system_prompt))


# Your role

Your are Finance Coach, a calm and practical study assistant who helps studnet review liquidity, working capital, cash management, and breakeven analysis Finance Coach  asks focused questions, explains formulas clearly, and helps the student identify topics they should review again.

#Context 

Here are the study notes you should use:

Module 5 Study Notes: Liquidity, Working Capital, and Breakeven
This study sheet summarizes Module 06: Liquidity, Working Capital, and Breakeven. The
main ideas are how entrepreneurs measure short-term financial health, manage cash and
working capital, and calculate the revenue level needed for survival or operating
breakeven.
Financial Ratios and Liquidity
Financial ratio analysis uses relationships among financial statement accounts to judge
a company’s condition and performance. Ratios can help entrepreneurs, lenders, and
investors see whether the business is becoming stronger or weaker over time.
Liquidity describes how easily assets can be converted into cash without losing value.
Cash is the most liquid asset. Receivables and marketable securities are also relatively
liquid. Fixed assets such as equipment, real estate, and specialized property are
usually less liquid because they take more time and effort to sell.
Accounting liquidity measures whether a company can meet short-term obligations with
liquid assets. A venture can be profitable but still face liquidity problems if cash is
tied up in inventory, receivables, or fixed assets.
Common Liquidity Measures
Current ratio = Current Assets / Current Liabilities. It shows whether current assets
could cover short-term liabilities. A current ratio above 1.0 usually means current
assets exceed current liabilities, but it does not guarantee all assets can quickly
become cash.
Quick ratio = (Current Assets - Inventory) / Current Liabilities. This is stricter than
the current ratio because it removes inventory, which may not be easy to sell quickly.
Cash ratio = Cash and Marketable Securities / Current Liabilities. This is the most
conservative liquidity ratio because it only counts funds that are immediately
available.
Net working capital = Current Assets - Current Liabilities. NWC is a dollar measure of
the company’s short-term cushion. NWC-to-total-assets ratio compares the working capital
buffer with the size of the company.
Cash Burn and Liquidity Events
Cash burn measures how quickly a venture uses cash. Monthly cash burn helps estimate how
long the company can continue operating with its available cash. If cash burn grows
faster than cash build, the company may become less liquid even while sales are
increasing.
A liquidity event allows investors or owners to cash out some or all ownership. Examples
include selling the company, selling to a strategic partner, going public, or selling an
interest to a private equity investor.
Working Capital Management
Page 1 of 3Working capital management is the management of short-term investment and financing. Its
goals are to maintain enough cash flow for operations and use resources productively.
Liquidity management means generating cash when and where it is needed. Drags on
liquidity delay cash collection, such as slow customer payments or obsolete inventory.
Pulls on liquidity cause cash to leave too soon, such as early payments to suppliers or
reduced access to credit.
Primary sources of liquidity include cash balances, short-term financing, trade credit,
bank loans, and cash flow management. Secondary sources include renegotiating debt,
selling assets, or bankruptcy reorganization.
Cash Conversion Cycle and Cash Management
The cash conversion cycle is the length of time it takes for inventory investment to
generate cash, considering that some inventory may be purchased on credit. Longer
operating and cash conversion cycles increase the need for liquidity.
Managers forecast short-term cash flows, monitor cash balances, and maintain target cash
balances to avoid negative cash positions. Cash inflows include customer receipts,
investment maturities, debt proceeds, and tax refunds. Cash outflows include payroll,
payables, investments, debt repayment, interest, dividends, and taxes.
Short-term investments can temporarily store excess funds. Managers consider liquidity,
maturity, credit risk, yield, and collateral requirements when choosing short-term
investments.
Receivables, Inventory, and Payables
Accounts receivable management focuses on efficient records, accurate collections,
communication with credit managers, and reporting. Companies may use credit terms such
as net days, cash before delivery, cash on delivery, bill-to-bill, or monthly billing.
An aging schedule shows how long receivables have been outstanding. Number of days
receivable should be compared with the company’s credit terms and competitors.
Inventory management seeks enough inventory to meet demand without holding more than
necessary. Motives for inventory include transaction needs, precaution against
stockouts, and speculation about price or availability. Common approaches include
economic order quantity, just-in-time systems, and materials/resource planning.
Accounts payable are a spontaneous form of credit from suppliers. Managers compare
number of days payable with credit terms and consider trade credit costs, vendor
relationships, alternative financing, and disbursement float.
Breakeven Terms
Variable expenses change directly with revenues, such as cost of goods sold. Fixed
expenses stay relatively constant over a relevant revenue range, such as administrative
costs, marketing costs, or some financing costs.
Variable Cost Revenue Ratio, or VCRR, equals Variable Costs / Revenues. Contribution
margin equals 1 - VCRR. A higher contribution margin means more of each revenue dollar
Page 2 of 3is available to cover fixed costs.
Cash flow breakeven means cash in equals cash out for a period. EBDAT means earnings
before depreciation, amortization, and taxes. EBDAT breakeven is the survival revenue
needed to cover cash operating expenses and fixed financing costs.
EBDAT and NOPAT Breakeven
EBDAT = Revenues - Variable Costs - Cash Fixed Costs. At EBDAT breakeven, EBDAT equals
zero, so Revenues = Variable Costs + Cash Fixed Costs.
Survival Revenues = Cash Fixed Costs / (1 - VCRR). For example, if cash fixed costs are
$400,000 and VCRR is 65%, then contribution margin is 35% and survival revenues are
about $1,143,000.
NOPAT means net operating profit after taxes. NOPAT breakeven focuses on operations
rather than financing. NOPAT breakeven revenues = Total Operating Fixed Costs / (1 -
VCRR). Total operating fixed costs include cash operating fixed costs plus noncash
operating costs such as depreciation, but exclude interest expense.
EBDAT breakeven is especially important for short-term survival because failure to cover
cash costs signals financial distress. Failure to cover noncash depreciation is not
necessarily an immediate cash crisis.
Key Takeaways
- Liquidity is about having enough usable cash or near-cash assets to meet short-term
obligations.
- The current ratio, quick ratio, cash ratio, and net working capital help diagnose
liquidity strength, but each has limitations.
- Working capital management focuses on cash, receivables, inventory, payables, short-term
financing, and the cash conversion cycle.
- Breakeven analysis shows how much revenue a venture needs to survive or cover operating
costs.
- Contribution margin is central to breakeven. If the contribution margin rises, the
revenue needed to break even falls.
Page 3 of 3

# Rules
Asl exactly one question at a time based only on the notes.

Wait for the student's answer before askign another question.

After the student answers, clearly say whether the answer is correct or incorrect, then give a vreif explanation.

If the student gets a question wrong, says "I don't know", or seems unsure, use your tool to record the topic or question they should review.

Do not ask questions about material that is not in the notes.

Keep your tone calm, practical, and encouraging



## 5. Test your Persona Agent with a single question

**TODO:** Copy the pattern from Lab 4, Step 6 — build a `messages` list with your `system_prompt`
and one user message like "I'm ready, let's start the quiz", and call the model.

In [6]:
messages = [
    {"role": "system", "content": system_prompt},
    {"role": "user", "content": "I'm ready, let's start the quiz"},
]

response = azure_client.chat.completions.create(
    model=DEPLOYMENT,
    messages=messages
)

response.choices[0].message.content

'Great — first question:\n\nWhat is the formula for the current ratio?'

## 6. Turn it into a `chat(message, history)` function

**TODO:** Copy `chat()` from Lab 4, Step 7 exactly. The only thing that's different is which
`system_prompt` it uses — and you've already built that above.

In [8]:
def chat(message, history):
    messages = [{"role": "system", "content": system_prompt}] + history + [{"role": "user", "content": message}]
    
    response = azure_client.chat.completions.create(model=DEPLOYMENT, messages=messages)
    
    return response.choices[0].message.content

In [9]:
chat("I'm ready, let's start the quiz", [])

'Great — first question:\n\nWhat is the formula for the current ratio?'

## 7. Give it a real chat UI with Gradio

This part needs no changes from Lab 4 — one line.

In [10]:
gr.ChatInterface(chat).launch(inbrowser=True)

# When you're done chatting, interrupt this cell (stop button) before moving on.

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


## 8. Give your Persona a tool

Now design **one tool** your Persona could plausibly use. Lab 4 logged missed questions to a
file — you can keep that same idea, or design something else, as long as it's a plain Python
function that does something simple and visible.

**Ideas:**
- Keep Lab 4's idea, adapted to your topic: `log_missed_question_tool(question)` — appends a
  missed question to a review file
- `track_score_tool(correct)` — keeps a running tally of right/wrong answers in a small text or
  JSON file
- `bookmark_topic_tool(topic)` — records a whole sub-topic (not just one question) that seems to
  need more review

**TODO:** Write your own tool function below, modeled on `log_missed_question_tool` from Lab 4,
Step 9. It should take one simple argument, do something small and visible, and return a short
string.

In [12]:
def my_tool(value):
    print(f"Tool called to log review items: {value}")
    with open("review_items.txt", "a", encoding="utf-8") as f:
        f.write(value + "\n")
    return "Review item logged"

In [13]:
my_tool("test value")  # quick manual test of your tool

Tool called to log review items: test value


'Review item logged'

## 9. Describe your tool to the model with JSON

**TODO:** Copy the JSON-schema pattern from Lab 4, Step 10, but describe *your* tool and *your*
argument name instead of `log_missed_question_tool`/`question`.

**Hint:** the shape never changes — only `name`, `description`, and the one property inside
`parameters` need to match your own tool.

In [14]:
my_tool_json = {
    "name": "my_tool",  # TODO: match your function's name
    "description": "Use this tool to record a topic or question the student should review again after they answer incorrectly, say they do not know, or seem unsure.",
    "parameters": {
        "type": "object",
        "properties": {
            "value": {"type": "string", "description": "The topic or quiz question the student should review again"}
        },
        "required": ["value"],
        "additionalProperties": False
    }
}

tools = [{"type": "function", "function": my_tool_json}]

## 10. Update `chat()` to handle one tool call

**TODO:** Copy the single-tool-call version of `chat()` from Lab 4, Step 11. You'll need to:
- Pass `tools=tools` into the API call
- Check `response.choices[0].finish_reason == "tool_calls"`
- Pull the tool call's arguments out with `json.loads(tool_call.function.arguments)`
- Call your own tool function instead of `log_missed_question_tool`
- Append the tool result back into `messages` with `role="tool"`, then call the model again

**Hint:** rename the response's message object to `message_obj` (not `message`) so it doesn't
clash with the `message` parameter of `chat()` — this tripped people up in Lab 4.

In [15]:
def chat(message, history):
    messages = [{"role": "system", "content": system_prompt}] + history + [{"role": "user", "content": message}]
    response = azure_client.chat.completions.create(model=DEPLOYMENT, messages=messages, tools=tools)

    if response.choices[0].finish_reason == "tool_calls":
        message_obj = response.choices[0].message
        tool_call = message_obj.tool_calls[0]

        arguments = json.loads(tool_call.function.arguments)
        value = arguments.get("value")

        result = my_tool(value)

        messages.append(message_obj)
        messages.append({
            "role": "tool",
            "tool_call_id": tool_call.id,
            "content": result
        })

        response = azure_client.chat.completions.create(
            model=DEPLOYMENT,
            messages=messages,
            tools=tools
        )

    return response.choices[0].message.content

In [17]:
gr.ChatInterface(chat).launch(inbrowser=True)

# Try to trigger your tool — get a question wrong on purpose, or whatever your tool responds to.
# Interrupt the cell when you're done.

* Running on local URL:  http://127.0.0.1:7862
* To create a public link, set `share=True` in `launch()`.


## 11. The agent loop: handle as many tool calls as needed

**TODO:** Upgrade `chat()` one more time, exactly like Lab 4, Step 12:
- Change `if` to `while`
- Loop over **all** of `message_obj.tool_calls` with a `for` loop, not just the first one

This is the same two changes you saw in Lab 4 — nothing new to invent here.

In [18]:
def chat(message, history):
    messages = [{"role": "system", "content": system_prompt}] + history + [{"role": "user", "content": message}]
    response = azure_client.chat.completions.create(model=DEPLOYMENT, messages=messages, tools=tools)

    while response.choices[0].finish_reason == "tool_calls":
        message_obj = response.choices[0].message
        messages.append(message_obj)

        for tool_call in message_obj.tool_calls:
            arguments = json.loads(tool_call.function.arguments)
            value = arguments.get("value")

            result = my_tool(value)

            messages.append({
                "role": "tool",
                "tool_call_id": tool_call.id,
                "content": result
            })

        response = azure_client.chat.completions.create(
            model=DEPLOYMENT,
            messages=messages,
            tools=tools
        )

    return response.choices[0].message.content

## 12. Try your finished Persona Agent
Same one line as before.

In [ ]:
gr.ChatInterface(chat).launch(inbrowser=True)

# Try a message that should trigger your tool more than once in a row (for example, admitting
# you got several past questions wrong at once). Interrupt the cell when you're done chatting.

* Running on local URL:  http://127.0.0.1:7863
* To create a public link, set `share=True` in `launch()`.


Tool called to log review items: Review the formula for the current ratio: Current Assets / Current Liabilities.
Tool called to log review items: Review the quick ratio formula: (Current Assets - Inventory) / Current Liabilities


## Reflection questions

Answer these in a new Markdown cell below.

1. What topic did you choose, and what tool did you give your Persona? Why did that tool make sense for this topic?
   - I chose my financial entreprenuership course as my topic. I gave my tool a pdf that had information about liquidty, working capital managment, and breakeven analysis. The tool writes missed topics to a text file which makes sense for a student or agent to call back on for future review.
2. Compare building this agent to building Study Buddy in Lab 4. What was exactly the same? What, if anything, did you have to think about differently?
   - My tool takes a review item as input, prints that it was called, then writes the item to the review_items.txt file. I tested it by answering questions incorrectly and asking the agent to log topics for review.
3. What would break if you asked your Persona a question completely unrelated to your notes? Did your system prompt handle that the way you wanted? 
 - The agent loop is important because sometimes the model will need to call on a tool before it completes the response. The loop lets the program sit with the tool calls until the tool results are sent back to the model to integrate into its response. 
4. If you added a second tool to this agent, what would it be, and what would change in Steps 9–11 to support it?
 -I would add a bookmark topic tool that saves the important topics for a student to revisit later. I would add a second tool function, add another JSON tool description and incude it in the tools list. 

## Wrap-up

Before submitting:

- Make sure your `.env` file is **not** included in what you submit — it contains your API key.
- Run the notebook top to bottom one more time to confirm it works cleanly.
- Answer all four reflection questions.
- Rename this file to include your name, e.g. `Assignment5_JaneDoe.ipynb`, and submit it.

You've now hand-built a quiz agent with a real tool and a real agent loop — the same shape used by
production agent systems, just without a framework hiding the details from you.